In [ ]:
# Colab setup: install pyedautils; elsewhere load plotly.js from a CDN for the plots
import sys
if "google.colab" in sys.modules:
    %pip install -q "pyedautils>=0.0.33"
else:
    import plotly.io as pio
    pio.renderers.default = "notebook_connected"

# Exploratory Data Analysis (EDA)

Exploratory Data Analysis is the most important step in any data project. Before building models or drawing conclusions, you need to **understand your data** — its structure, quality, patterns, and anomalies.

For sensor data, EDA typically answers questions like:
- Are there gaps or outliers in the sensor readings?
- What are the daily and seasonal patterns?
- How do different variables relate to each other (e.g., outdoor temperature vs. heating energy)?
- Has something changed over time (e.g., HVAC system replacement, occupancy changes)?

This chapter walks through a systematic EDA workflow using a real temperature and humidity dataset from four flats.

## EDA Workflow Overview

The first four steps are shown here; the deeper analyses each have their own recipe in Part III, linked in the [checklist](eda-checklist) at the end.

::::{grid} 2 2 4 4
:gutter: 2

:::{grid-item-card} Step 1
:link: step-1-load-and-inspect
:link-type: ref
**Load & Inspect**

`shape`, `info()`, `describe()`
:::

:::{grid-item-card} Step 2
:link: step-2-data-quality
:link-type: ref
**Data Quality**

Missing values, duplicates, gaps
:::

:::{grid-item-card} Step 3
:link: step-3-time-series-overview
:link-type: ref
**Time Series Overview**

Plot raw data, zoom in
:::

:::{grid-item-card} Step 4
:link: step-4-distribution-analysis
:link-type: ref
**Distribution Analysis**

Histograms
:::

::::

(step-1-load-and-inspect)=
## Step 1: Load and Inspect

Always start by loading the data, parsing dates, and getting a first overview. If you are unfamiliar with loading data in pandas, see the [Loading Data](loadingData) chapter first.

In [ ]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/retomarek/edap/main/edap/sampleData/flatTempHum.csv"

df = pd.read_csv(url, sep=";")
df["time"] = pd.to_datetime(df["time"])
df = df.set_index("time")

print(f"Shape: {df.shape}")
print(f"Period: {df.index.min()} to {df.index.max()}")
print(f"Duration: {df.index.max() - df.index.min()}")
df.head()

In [ ]:
df.describe()

:::{admonition} What to look for in describe()
:class: tip dropdown
- **count** differs between columns? → some columns have more missing values
- **min/max** outside physically plausible range? → possible sensor error (e.g., temperature of -999 or 99)
- **mean vs. 50%** (median) very different? → skewed data, possibly caused by outliers
- **std** very large relative to mean? → high variability or mixed data
:::

(step-2-data-quality)=
## Step 2: Data Quality

Real-world sensor data almost always has quality issues: missing values, duplicate timestamps, or irregular intervals. Identifying these early prevents wrong conclusions later.

In [ ]:
# Missing values per column
missing = pd.DataFrame({
    "missing": df.isna().sum(),
    "% missing": (df.isna().mean() * 100).round(1)
})
missing

In [ ]:
from pyedautils.data_quality import plot_missing_values_heatmap

fig = plot_missing_values_heatmap(df)
fig.show()

:::{admonition} Interpreting the missing values heatmap
:class: tip dropdown
Red bands indicate periods where sensors reported no data. Common causes:
- **Vertical red band across all sensors** → system-wide outage (network, gateway, power)
- **Single sensor red band** → individual sensor failure or battery empty
- **Red at the end** → sensors stopped reporting (decommissioned or data export cutoff)
:::

In [ ]:
# Check for duplicate timestamps and irregular intervals
print(f"Duplicate timestamps: {df.index.duplicated().sum()}")
print(f"Index is monotonic:  {df.index.is_monotonic_increasing}")

# Most common interval
time_diffs = df.index.to_series().diff().dropna()
print(f"Most common interval: {time_diffs.mode()[0]}")
print(f"Min interval: {time_diffs.min()}")
print(f"Max interval: {time_diffs.max()}")

(step-3-time-series-overview)=
## Step 3: Time Series Overview

Plot the raw data to get a visual overview. Interactive plots allow zooming into specific periods.

In [ ]:
from pyedautils.plots import plot_timeseries

fig = plot_timeseries(df, columns=["FlatA_Temp"], title="FlatA Temperature", ylab="Temperature [°C]")
fig.show()

# Column list reused in later cells
temp_cols = [col for col in df.columns if "Temp" in col]

In [ ]:
fig = plot_timeseries(df, columns=temp_cols, title="All Flats — Temperature", ylab="Temperature [°C]")
fig.show()

(step-4-distribution-analysis)=
## Step 4: Distribution Analysis

A histogram reveals the **distribution** of values — how often each temperature occurs and whether there are unusual clusters. Box plots per month or hour show how the distribution changes over time (see [Boxplot](../partDataVis/statisticalBoxplot.ipynb)).

In [ ]:
from pyedautils.plots import plot_distribution

fig = plot_distribution(df, columns=temp_cols, title="Temperature Distribution by Flat", xlab="Temperature [°C]")
fig.show()

:::{admonition} What distribution patterns tell you
:class: tip dropdown
- **Two peaks** (bimodal) → different operating modes (e.g., heating season and summer, occupied/unoccupied)
- **Long tail on one side** → occasional events (window openings, sun) or sensor problems
- **One flat with a very narrow distribution** → heavily controlled room or a stuck sensor
:::

Steps 5 to 8 (outliers, correlation, decomposition, autocorrelation) each have their own recipe in Part III; the checklist below links them.

(eda-checklist)=
## EDA Checklist

Use this checklist as a starting point for every new dataset:

| Step | Question | Where in this book |
|---|---|---|
| **1. Inspect** | How big? What columns? What types? | `shape`, `dtypes`, `info()`, `describe()` (above) |
| **2. Quality** | Missing values? Duplicates? Gaps? | [Data Quality](dataQuality.ipynb) |
| **3. Visualize** | What does the raw data look like? | `plot_timeseries()` (above) |
| **4. Distribute** | How are values distributed? | [Histogram](../partDataVis/statisticalDensity.ipynb), [Boxplot](../partDataVis/statisticalBoxplot.ipynb) |
| **5. Outliers** | Are there suspicious values? | [Outlier Detection](../partDataVis/statisticalOutliers.ipynb) |
| **6. Correlate** | How do variables relate? | [Correlation & Scatter](../partDataVis/statisticalCorrelation.ipynb), [Cross-Correlation](../partDataVis/statisticalCrossCorrelation.ipynb) |
| **7. Decompose** | What are the trend and seasonal patterns? | [Time Series Decomposition](../partDataVis/decomposition.md) |
| **8. Cycles** | What are the dominant cycles? | [Autocorrelation](../partDataVis/statisticalAutocorrelation.ipynb) |